In [1]:
import os
import json

def convert_directory_to_json(root_dir):
    """
    将文件目录转换为 JSON 格式。
    :param root_dir: 文件目录的根路径
    :return: 文件目录的 JSON 表示
    """
    def parse_directory(directory_path):
        """递归解析文件夹结构"""
        items = []
        for item in os.listdir(directory_path):
            item_path = os.path.join(directory_path, item)
            if os.path.isdir(item_path):
                items.append({
                    "type": "directory",
                    "name": item,
                    "path": item_path,
                    "children": parse_directory(item_path)  # 递归解析子文件夹
                })
            elif os.path.isfile(item_path):
                items.append({
                    "type": "file",
                    "name": item,
                    "path": item_path
                })
        return items

    # 解析根目录
    json_data = {
        "root_directory": root_dir,
        "contents": parse_directory(root_dir)
    }
    return json_data

# 示例使用
root_dir = "BraTS2021_Training_Data"
json_data = convert_directory_to_json(root_dir)

# 保存为 JSON 文件
with open("directory_structure.json", "w") as f:
    json.dump(json_data, f, indent=4)

print("文件目录已成功转换为 JSON 文件：directory_structure.json")

文件目录已成功转换为 JSON 文件：directory_structure.json


In [12]:

import requests
import csv
import re
import json
import os
# 如果config.py存在的话，从config.py导入API_KEY
if os.path.exists("config.py"):
    from config import API_KEY

# DeepSeek API 的 URL 和 API 密钥
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"

# 从环境变量中读取 API 密钥
API_KEY = API_KEY

# 读取 JSON 文件
with open("directory_structure.json", "r") as f:
    json_input = f.read()

# 构建请求数据
data = {
    "model": "deepseek-chat",
    "messages": [
        {"role": "system", "content": "你是一个数据分析助手，能够解析文件目录并生成结构化的 JSON 输出。"},
        {"role": "user", "content": f"""
        请解析以下 JSON 文件目录，并按照样本归类。输出格式为 JSON，包含以下字段：
        - sample_id: 样本的唯一标识符
        - modalities: 样本包含的模态图像（如 flair, t1, t1ce, t2）
        - has_segment_mask: 是否存在分割掩码（如 seg）

        示例输出：
        [
            {{
                "sample_id": "conclude from the name",
                "modalities (or modality)": ["if not clear make it a number"],
                "has_segment_mask": true
            }},
            {{
                "sample_id": "conclude from the name",
                "modalities (or modality)": ["if not clear make it a number"],
                "has_segment_mask": true
            }}
        ]

        请解析以下 JSON 文件目录：
        {json_input}
        """}
    ],
    "stream": False
}

# 发送请求
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}
response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data)

# 检查响应状态码
if response.status_code == 200:
    result = response.json()
    try:
        # 打印模型输出的原始内容
        print("模型输出的原始内容：")
        model_output = result["choices"][0]["message"]["content"]
        print(model_output)

        output_json = re.sub(r"```json|```", "", model_output).strip()
        with open("result.json", "w") as f:
            f.write(output_json)
        print("模型输出的原始内容已保存到 result.json 文件中。")
    except json.JSONDecodeError:
        print("模型的输出不是有效的 JSON 格式：")
        print(result["choices"][0]["message"]["content"])
else:
    print(f"请求失败，状态码：{response.status_code}")
    print(response.text)

模型输出的原始内容：
```json
[
    {
        "sample_id": "BraTS2021_00000",
        "modalities": ["flair", "t1", "t1ce", "t2"],
        "has_segment_mask": true
    },
    {
        "sample_id": "BraTS2021_00002",
        "modalities": ["flair", "t1", "t1ce", "t2"],
        "has_segment_mask": true
    },
    {
        "sample_id": "BraTS2021_00003",
        "modalities": ["flair", "t1", "t1ce", "t2"],
        "has_segment_mask": true
    }
]
```
模型输出的原始内容已保存到 result.json 文件中。
